# Day 15 — The Ultralytics Framework on T4

**Agri-AI Bootcamp** · Phase 4 of 5: *Object Detection & Yield* · Days 13–16

> Install ultralytics, load YOLOv8n and run inference on a farm image or video.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

Ultralytics packages YOLOv8 for loading, predicting and training. Model sizes (n, s, m, l, x) trade accuracy for speed and memory; nano is the safe choice on a free T4. Results give boxes, classes and confidences per image.

### 🌾 Agriculture example

Run yolov8n.pt on a farm photo to find standard classes such as cow, sheep or truck; tractors and crop diseases need custom training.

> **Watch out:** Pre-trained COCO weights don’t know your crops; Day 16 fixes that.

## 2. Setup

In [ ]:
!pip -q install ultralytics

In [ ]:
from ultralytics import YOLO
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
import ultralytics
print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())
ultralytics.checks()

## 3. Data

In [ ]:
# ---- An image to run on ------------------------------------------------------
# Option A: upload your own farm photo
#   from google.colab import files; up = files.upload(); IMG = list(up)[0]
# Option B: use a sample image that ships with ultralytics (no download needed)
from pathlib import Path
import urllib.request, os

IMG = "farm.jpg"
if not os.path.exists(IMG):
    try:
        urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", IMG)
        print("downloaded a sample photo ->", IMG)
    except Exception as e:
        # offline fallback: synthesise a picture so the rest of the notebook runs
        print("no network, generating a placeholder image:", str(e)[:60])
        a = (np.random.default_rng(0).uniform(.3, .7, (480, 640, 3)) * 255).astype(np.uint8)
        Image.fromarray(a).save(IMG)
Image.open(IMG).resize((320, 240))

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
!pip -q install ultralytics
from ultralytics import YOLO
r=YOLO("yolov8n.pt")("farm.jpg")
r[0].show()
```

## 5. Worked solution

In [ ]:
# ---- Model sizes: accuracy vs speed vs memory -------------------------------
# n(ano) s(mall) m(edium) l(arge) x(tra).  On a free T4, start with n.
model = YOLO("yolov8n.pt")            # downloads ~6 MB of weights once
print("task:", model.task, "| classes:", len(model.names))
print("a few COCO classes:", {k: model.names[k] for k in list(model.names)[:8]})

# ---- Inference ---------------------------------------------------------------
results = model(IMG, conf=0.25, verbose=False)
r = results[0]
print(f"\n{len(r.boxes)} detections")
for b in r.boxes:
    cls  = model.names[int(b.cls)]
    conf = float(b.conf)
    x1, y1, x2, y2 = b.xyxy[0].tolist()
    print(f"  {cls:<12} conf {conf:.2f}  box ({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})")

plt.figure(figsize=(7, 5.5))
plt.imshow(r.plot()[:, :, ::-1])       # r.plot() returns BGR
plt.axis("off"); plt.title("YOLOv8n on COCO classes"); plt.show()

# ---- What is inside a Results object ----------------------------------------
print("\nboxes.xyxy  :", tuple(r.boxes.xyxy.shape), "corner coordinates")
print("boxes.xywhn :", tuple(r.boxes.xywhn.shape), "centre/size, normalised 0-1")
print("boxes.conf  :", r.boxes.conf.tolist())
print("boxes.cls   :", [model.names[int(c)] for c in r.boxes.cls])
print("speed (ms)  :", {k: round(v, 1) for k, v in r.speed.items()})

# ---- The honest limit of pre-trained weights --------------------------------
AG_WORDS = ["cow", "sheep", "horse", "bird", "truck", "person", "potted plant"]
have = [n for n in model.names.values() if n in AG_WORDS]
print(f"\nCOCO does cover: {have}")
print("COCO does NOT cover: tractor, wheat head, tomato, weed species, leaf lesions.")
print("Counting fruit or spotting a disease needs custom training -- that is Day 16.")

# ---- conf and iou are the two knobs you will actually turn ------------------
print("\nconf   detections")
for c in [0.1, 0.25, 0.5, 0.75]:
    n = len(model(IMG, conf=c, verbose=False)[0].boxes)
    print(f"{c:<6} {n}")

## 6. Your turn

1. Run the same image through yolov8s.pt and compare detections and inference time.
2. Run on a short video with `model.predict(source='clip.mp4', save=True)` and look at the saved file.
3. Set `classes=[19, 20]` to keep only cow and sheep, and confirm nothing else appears.

In [ ]:
# TODO 1: nano vs small
import time
for w in ["yolov8n.pt", "yolov8s.pt"]:
    m = YOLO(w); t0 = time.time()
    res = m(IMG, conf=0.25, verbose=False)
    print(f"{w}: {len(res[0].boxes)} boxes in {time.time()-t0:.2f}s, "
          f"{sum(p.numel() for p in m.model.parameters())/1e6:.1f}M params")

# TODO 2: video
# model.predict(source="clip.mp4", save=True, conf=0.3)

# TODO 3: filter classes
# model(IMG, classes=[19, 20], verbose=False)[0].boxes

## 7. Dataset and references

**Dataset:** [Roboflow Universe: search fruit / pepper / weed detection](https://universe.roboflow.com)

**References**

- [Ultralytics docs](https://docs.ultralytics.com)

**Course material:** Ultralytics YOLOv8 documentation

---
**Next:** Day 16 — Custom Ag Object Detection